## Install required packages

In [ ]:
!pip install torch transformers datasets pandas numpy scikit-learn matplotlib seaborn

## Import the libraries

In [ ]:
import os, time, json, math, random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_fscore_support
from collections import Counter
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

## Set the variables and the seed

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROBERTA_MODEL_NAME = 'microsoft/deberta-v3-base'
FINETUNED_MODEL_PATH = 'sentiment_finetuned_model'

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='cuda')

## Reading the data

In [ ]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSupomyu2ggRui1SIB89zSxqkz_LIMPyaHNG7lFJf6B8NYSzj6kOR76nuKrDgDeJL7LSQjdvFBwVyyP/pub?gid=915603681&single=true&output=csv'
df = pd.read_csv(url, encoding='utf-8')

## Renaming the variables, dropping the null and typecasting to int

In [ ]:
df = df.rename(columns={'sentence':'text', 'sentiment':'label'})
df = df[['text', 'label']].dropna()
df['label'] = df['label'].astype(int)

## Getting the labels and printing the distribution

In [ ]:
label_names = {0: 'Negative', 1: 'Neutral/Mixed', 2: 'Positive'}

print(f"Total rows: {len(df)}")
print("Label distribution:")
for k, v in sorted(Counter(df['label']).items()):
    print(f"  {label_names[k]} ({k}): {v} ({v/len(df):.1%})")

Total rows: 16016
Label distribution:
  Negative (0): 5529 (34.5%)
  Neutral/Mixed (1): 4294 (26.8%)
  Positive (2): 6193 (38.7%)


## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.15,
    random_state=SEED,
    stratify=df['label']
)

# From training, carve out 10% for validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.10,
    random_state=SEED,
    stratify=y_train
)

print(f"Train: {len(X_tr)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 12251 | Val: 1362 | Test: 2403


## Loading the model and tokenizing the dataset

In [ ]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_df = pd.DataFrame({'text': X_tr, 'label': y_tr})
val_df   = pd.DataFrame({'text': X_val, 'label': y_val})
test_df  = pd.DataFrame({'text': X_test, 'label': y_test})

ds_train = Dataset.from_pandas(train_df)
ds_val   = Dataset.from_pandas(val_df)
ds_test  = Dataset.from_pandas(test_df)

MAX_LEN = 256

def tok_fn(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN
    )

tok_train = ds_train.map(tok_fn, batched=True)
tok_val   = ds_val.map(tok_fn, batched=True)
tok_test  = ds_test.map(tok_fn, batched=True)

cols = ['input_ids', 'attention_mask', 'label']
tok_train.set_format(type='torch', columns=cols)
tok_val.set_format(type='torch', columns=cols)
tok_test.set_format(type='torch', columns=cols)

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/12251 [00:00<?, ? examples/s]

Map:   0%|          | 0/1362 [00:00<?, ? examples/s]

Map:   0%|          | 0/2403 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
).to(device)


bs_train = 16
bs_eval  = 32

training_args = TrainingArguments(
    output_dir="./deberta_results",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=bs_train,
    per_device_eval_batch_size=bs_eval,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,  # mixed precision on T4
    logging_steps=50,
    warmup_steps=100,
    weight_decay=0.01,
    metric_for_best_model="f1",
    report_to=[],               # disable W&B/TensorBoard logs
    dataloader_num_workers=0    # conservative for notebook
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=None)
    labelwise = {}
    for idx, lbl in enumerate(sorted(set(labels))):
        name = label_names.get(lbl, str(lbl))
        labelwise[name] = {
            'precision': float(precision[idx]),
            'recall': float(recall[idx]),
            'f1_score': float(f1[idx])
        }
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "labelwise": labelwise
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Labelwise
1,0.586300,0.515794,0.795154,0.795822,"{'Negative': {'precision': 0.8011928429423459, 'recall': 0.8574468085106383, 'f1_score': 0.828365878725591}, 'Neutral/Mixed': {'precision': 0.6521739130434783, 'recall': 0.6575342465753424, 'f1_score': 0.654843110504775}, 'Positive': {'precision': 0.8961303462321792, 'recall': 0.8349146110056926, 'f1_score': 0.8644400785854617}}"
2,0.428500,0.533493,0.799559,0.802266,"{'Negative': {'precision': 0.8260869565217391, 'recall': 0.8489361702127659, 'f1_score': 0.8373557187827911}, 'Neutral/Mixed': {'precision': 0.6384039900249376, 'recall': 0.7013698630136986, 'f1_score': 0.6684073107049608}, 'Positive': {'precision': 0.9079497907949791, 'recall': 0.8235294117647058, 'f1_score': 0.863681592039801}}"
3,0.330900,0.587094,0.805433,0.806516,"{'Negative': {'precision': 0.8408602150537634, 'recall': 0.8319148936170213, 'f1_score': 0.8363636363636363}, 'Neutral/Mixed': {'precision': 0.6614173228346457, 'recall': 0.6904109589041096, 'f1_score': 0.675603217158177}, 'Positive': {'precision': 0.8798449612403101, 'recall': 0.8614800759013282, 'f1_score': 0.8705656759348035}}"


TrainOutput(global_step=2298, training_loss=0.4768369311970976, metrics={'train_runtime': 1043.3722, 'train_samples_per_second': 35.225, 'train_steps_per_second': 2.202, 'total_flos': 4835190432084480.0, 'train_loss': 0.4768369311970976, 'epoch': 3.0})

In [ ]:
trainer.save_model('./deberta_model')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy a file to your Google Drive
!cp -r ./deberta_model /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
